Thanks to 

@CRODOC notebook: https://www.kaggle.com/code/crodoc/82409-improved-baseline-santa-2022

@DANIEL PRECIADO notebook!: // https://www.kaggle.com/code/jazivxt/tsp-cost-function-start
    
@FILIP STRZAŁKA!:    https://www.kaggle.com/code/snufkin77/further-point-order-improvements-tsp-start
        
@ NICU !:           https://www.kaggle.com/code/nicupetridean/further-analysis-of-costs-points-ordering

In [ ]:
import numpy as np
import pandas as pd
from functools import reduce
pd.options.mode.chained_assignment = None
import seaborn as sns

p = '/kaggle/input/santa-2022/'
sub = pd.read_csv(p+'sample_submission.csv')
df_image = pd.read_csv(p+'image.csv')

In [ ]:
df_image

In [ ]:
sns.countplot(df_image['x'][:10])

In [ ]:
sns.countplot(df_image['y'][:4500])

In [ ]:
def df_to_image(df):
    side = int(len(df) ** 0.5)  # assumes a square image
    return df_image.set_index(['x', 'y']).to_numpy().reshape(side, side, -1)

In [ ]:
image = df_to_image(df_image)

In [ ]:
import cv2
import matplotlib.pyplot as plt
plt.plot(image[3])
plt.show()

In [ ]:
def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

In [ ]:
def cartesian_to_array(x, y, shape):
    m, n = shape[:2]
    i = (n - 1) // 2 - y
    j = (n - 1) // 2 + x
    if i < 0 or i >= m or j < 0 or j >= n:
        raise ValueError("Coordinates not within given dimensions.")
    return i, j

In [ ]:
df = sub[:-1]

In [ ]:
df

In [ ]:
df['path'] = df['configuration'].map(lambda x: [list(map(int, p2.split(' '))) for p2 in x.split(';')])

In [ ]:
df['path']

In [ ]:
df['point'] = df['path'].map(lambda x: get_position(x))

In [ ]:
df['point']

In [ ]:
df['image_point'] = df['path'].map(lambda x: cartesian_to_array(*get_position(x), image.shape))

In [ ]:
df['image_point']

In [ ]:
df['color'] = df['image_point'].map(lambda x : image[x])

In [ ]:
df['color'] 

In [ ]:
df['path'] = df['path'].map(lambda x: np.asarray(x))

In [ ]:
df['path']

In [ ]:
print(len(df), 257*257)

In [ ]:
df = df.drop_duplicates(subset=['image_point'], keep='first')


In [ ]:
df

In [ ]:
print(len(df)) #Some configurations may be helful in reducing load


In [ ]:
df.head()


In [ ]:
#modified cost functions

def reconfiguration_cost2(from_config, to_config):
    diffs = np.abs(from_config - to_config).sum(axis=1)
    return np.sqrt(diffs.sum())

def step_cost2(from_config, to_config, from_color, to_color):
    return reconfiguration_cost2(from_config, to_config) + (np.abs(to_color - from_color).sum() * 3)

In [ ]:
test = df[['path','color']].values


In [ ]:
%%time
d_sc = {}
for i in range(2): #range(len(df)):
    si = str(i)+','
    p1 = test[i][0]
    c1 = test[i][1]
    for j in range(len(df)):
        #d_sc[str(i) + ',' + str(j)] = step_cost2(df.path[i], df.path[j], df.color[i], df.color[j])
        d_sc[si + str(j)] = step_cost2(p1, test[j][0], c1, test[j][1])
    if i % 1000 == 0:
        print(i)
print([[c, d_sc[c]] for c in d_sc][:10]) #Idealy shouldn't have to use full range just withing small range of initial point or e.g., 5x5 and the rest can be coded as 99 or something

In [ ]:
#Worst case approximate hours on single core for a 17_849_761_609 dictionary or 133603 x 133603 matrix without reduction option
(len(df) * 1.84) / 60 / 60

In [ ]:
def compress_path(path):
    r = [[] for _ in range(8)]
    for p in path:
        for i in range(8):
            if len(r[i]) == 0 or r[i][-1] != p[i]:
                r[i].append(p[i])
    mx = max([len(x) for x in r])
    
    for rr in r:
        while len(rr) < mx:
            rr.append(rr[-1])
    r = list(zip(*r))
    for i in range(len(r)):
        r[i] = list(r[i])
    return r

In [ ]:
#From https://www.kaggle.com/code/ryanholbrook/getting-started-with-santa-2022
def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

def rotate_link(vector, direction):
    x, y = vector
    if direction == 1:  # counter-clockwise
        if y >= x and y > -x:
            x -= 1
        elif y > x and y <= -x:
            y -= 1
        elif y <= x and y < -x:
            x += 1
        else:
            y += 1
    elif direction == -1:  # clockwise
        if y > x and y >= -x:
            x += 1
        elif y >= x and y < -x:
            y += 1
        elif y < x and y <= -x:
            x -= 1
        else:
            y -= 1
    return (x, y)

def rotate(config, i, direction):
    config = config.copy()
    config[i] = rotate_link(config[i], direction)
    return config

def get_square(link_length):
    link = (link_length, 0)
    coords = [link]
    for _ in range(8 * link_length - 1):
        link = rotate_link(link, direction=1)
        coords.append(link)
    return coords

def get_neighbors(config):
    nhbrs = (
        reduce(lambda x, y: rotate(x, *y), enumerate(directions), config)
        for directions in product((-1, 0, 1), repeat=len(config))
    )
    return list(filter(lambda c: c != config, nhbrs))

def reconfiguration_cost(from_config, to_config):
    diffs = np.abs(np.asarray(from_config) - np.asarray(to_config)).sum(axis=1)
    return np.sqrt(diffs.sum())

def color_cost(from_position, to_position, image, color_scale=3.0):
    return np.abs(image[to_position] - image[from_position]).sum() * color_scale

def step_cost(from_config, to_config, image):
    from_position = cartesian_to_array(*get_position(from_config), image.shape)
    to_position = cartesian_to_array(*get_position(to_config), image.shape)
    return (
        reconfiguration_cost(from_config, to_config) +
        color_cost(from_position, to_position, image)
    )

def get_direction(u, v):
    """Returns the sign of the angle from u to v."""
    direction = np.sign(np.cross(u, v))
    if direction == 0 and np.dot(u, v) < 0:
        direction = 1
    return direction

# We don't use this elsewhere, but you might find it useful."""
def get_angle(u, v):
    """Returns the angle (in degreess) from u to v."""
    return np.degrees(np.math.atan2(
        np.cross(u, v),
        np.dot(u, v),
    ))

def get_path_to_point(config, point):
    """Find a path of configurations to `point` starting at `config`."""
    path = [config]
    # Rotate each link, starting with the largest, until the point can
    # be reached by the remaining links. The last link must reach the
    # point itself.
    for i in range(len(config)):
        link = config[i]
        base = get_position(config[:i])
        relbase = (point[0] - base[0], point[1] - base[1])
        position = get_position(config[:i+1])
        relpos = (point[0] - position[0], point[1] - position[1])
        radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
        # Special case when next-to-last link lands on point. 
        if radius == 1 and relpos == (0, 0):
            config = rotate(config, i, 1)
            if get_position(config) == point:
                path.append(config)
                break
            else:
                continue
        while np.max(np.abs(relpos)) > radius:
            direction = get_direction(link, relbase)
            config = rotate(config, i, direction)
            path.append(config)
            link = config[i]
            base = get_position(config[:i])
            relbase = (point[0] - base[0], point[1] - base[1])
            position = get_position(config[:i+1])
            relpos = (point[0] - position[0], point[1] - position[1])
            radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
    assert get_position(path[-1]) == point
                                                                                   
    path = compress_path(path)
    
    return path

def get_path_to_configuration(from_config, to_config):
    path = [from_config]
    config = from_config.copy()
    while config != to_config:
        for i in range(len(config)):
            config = rotate(config, i, get_direction(config[i], to_config[i]))
        path.append(config)
    assert path[-1] == to_config
    return path

# Compute total cost of path over image
def total_cost(path, image):
    return reduce(
        lambda cost, pair: cost + step_cost(pair[0], pair[1], image),
        zip(path[:-1], path[1:]),
        0,
    )

In [ ]:
from tqdm import tqdm

origin = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]
n = origin[0][0] * 2
board = []
flag = False
for split in range(2):
    for i in reversed(range(257)) if split%2==1 else range(257):
        if not flag:
            for j in range(128*split, 128+129*split):
                board.append((j-128,i-128))
        else:
            for j in reversed(range(128*split, 128+129*split)):
                board.append((j-128,i-128))
        flag = not flag
    flag = not flag

points = board[:]
print(len(points))

#Add Step Cost
#Add Neighbors
visited = []
path = [origin]
for p in tqdm(points):
    config = path[-1]
    if p not in visited:
        candy_cane_road = get_path_to_point(config, p)[1:]
        if len(candy_cane_road)>0:
            visited += [get_position(r) for r in candy_cane_road]
        path.extend(candy_cane_road)
path.extend(get_path_to_configuration(path[-1], origin)[1:])
total_cost(path, image)

In [ ]:
image = df_to_image(df_image)

total_cost(path, image)

In [ ]:
def config_to_string(config):
    return ';'.join([' '.join(map(str, vector)) for vector in config])

submission = pd.Series(
    [config_to_string(config) for config in path],
    name="configuration",
)

submission.to_csv('submission.csv', index=False)